# VQ-VAE

In [8]:
"""
VQ-VAE (Vector-Quantized VAE) interview scaffolding — PyTorch

Goal: give a candidate a runnable-ish skeleton with deliberate TODOs.
You can ask them to:
  1) implement VectorQuantizer forward (nearest-neighbor + losses)
  2) implement straight-through estimator
  3) wire losses + train loop
  4) verify shapes + debug common pitfalls

Works best for images (e.g., CIFAR-10 / MNIST). Swap dataset as desired.
"""

from dataclasses import dataclass
from typing import Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


# ----------------------------
# Config
# ----------------------------

@dataclass
class VQVAEConfig:
    in_channels: int = 3
    hidden_channels: int = 128
    z_channels: int = 64              # encoder output channels BEFORE quantization
    embedding_dim: int = 64           # codebook vector dim (usually == z_channels)
    num_embeddings: int = 512         # codebook size
    commitment_cost: float = 0.25     # beta in VQ-VAE
    lr: float = 2e-4
    batch_size: int = 128
    num_workers: int = 2
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------
# Encoder / Decoder (simple conv stacks)
# ----------------------------

class Encoder(nn.Module):
    """
    Maps x: (B, C, H, W) -> z_e: (B, D, H', W')
    """
    def __init__(self, in_channels: int, hidden: int, z_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden, kernel_size=4, stride=2, padding=1),  # /2
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=4, stride=2, padding=1),       # /4
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, z_channels, kernel_size=3, stride=1, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x) # it's just passing the input through


class Decoder(nn.Module):
    """
    Maps z_q: (B, D, H', W') -> x_hat: (B, C, H, W)
    """
    def __init__(self, out_channels: int, hidden: int, z_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(z_channels, hidden, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, hidden, kernel_size=4, stride=2, padding=1),  # *2
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, out_channels, kernel_size=4, stride=2, padding=1),  # *4
            # Output activation depends on data scaling: sigmoid for [0,1], tanh for [-1,1]
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

In [ ]:
# ----------------------------
# Vector Quantizer (candidate implements)
# ----------------------------

class VectorQuantizer(nn.Module):
    """
    Standard VQ-VAE codebook quantizer.

    Input z_e: (B, D, H, W)
    Output:
      z_q_st: quantized with straight-through gradient, same shape as z_e
      info dict: indices, losses, perplexity, etc.

    Interview TODOs:
      - compute distances efficiently
      - pick nearest embedding for each latent position
      - compute VQ loss + commitment loss
      - implement straight-through estimator: z_q_st = z_e + (z_q - z_e).detach()
    """
    def __init__(self, num_embeddings: int, embedding_dim: int, commitment_cost: float):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        # Initialize embeddings (common choices: uniform or normal)
        nn.init.uniform_(self.embedding.weight, -1.0 / num_embeddings, 1.0 / num_embeddings)

    def forward(self, z_e: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        z_e: (B, D, H, W)
        """
        assert z_e.dim() == 4, f"expected (B,D,H,W), got {tuple(z_e.shape)}"
        B, D, H, W = z_e.shape
        assert D == self.embedding_dim, f"D={D} must equal embedding_dim={self.embedding_dim}"

        # Flatten to (N, D) where N = B*H*W
        z_e_flat = z_e.permute(0, 2, 3, 1).contiguous().view(-1, D)  # (N, D)

        # ---- TODO(1): compute nearest embedding indices ----
        # Hint: squared L2 distance:
        #   ||x - e||^2 = ||x||^2 + ||e||^2 - 2 x·e
        # z_e_flat: (N,D), embed: (K,D)
        # distances: (N,K)
        # indices: (N,)
        #
        # distances = ...
        # indices = distances.argmin(dim=1)
        #
        # For interview: ask them about memory; K might be large.
        embed = self.embedding.weight  # (K, D)

        # Placeholder (wrong) — candidate replaces:
        distances = torch.cdist(z_e_flat, embed) ** 2  # (N, K) (slow but easy)
        indices = distances.argmin(dim=1)              # (N,)

        # ---- TODO(2): lookup quantized vectors ----
        z_q_flat = self.embedding(indices)             # (N, D)
        z_q = z_q_flat.view(B, H, W, D).permute(0, 3, 1, 2).contiguous()  # (B,D,H,W)

        # ---- TODO(3): losses ----
        # VQ-VAE loss terms typically:
        #   codebook_loss = ||sg[z_e] - z_q||^2
        #   commitment_loss = beta * ||z_e - sg[z_q]||^2
        # where sg[] is stop-gradient (detach).
        codebook_loss = F.mse_loss(z_q, z_e.detach())
        commitment_loss = self.commitment_cost * F.mse_loss(z_e, z_q.detach())
        vq_loss = codebook_loss + commitment_loss

        # ---- TODO(4): straight-through estimator ----
        z_q_st = z_e + (z_q - z_e).detach()

        # Optional metrics: perplexity (how many codes are used)
        one_hot = F.one_hot(indices, num_classes=self.num_embeddings).float()  # (N, K)
        avg_probs = one_hot.mean(dim=0)                                        # (K,)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        info = {
            "indices": indices.view(B, H, W),  # nicer shape
            "vq_loss": vq_loss,
            "codebook_loss": codebook_loss,
            "commitment_loss": commitment_loss,
            "perplexity": perplexity,
        }
        return z_q_st, info

In [10]:
# ----------------------------
# VQ-VAE Model
# ----------------------------

class VQVAE(nn.Module):
    def __init__(self, cfg: VQVAEConfig):
        super().__init__()
        self.encoder = Encoder(cfg.in_channels, cfg.hidden_channels, cfg.embedding_dim)
        self.quantizer = VectorQuantizer(cfg.num_embeddings, cfg.embedding_dim, cfg.commitment_cost)
        self.decoder = Decoder(cfg.in_channels, cfg.hidden_channels, cfg.embedding_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        z_e = self.encoder(x)              # (B,D,H',W')
        z_q, qinfo = self.quantizer(z_e)   # (B,D,H',W')
        x_hat = self.decoder(z_q)          # (B,C,H,W)
        out = {"z_e": z_e, "z_q": z_q, **qinfo}
        return x_hat, out


# ----------------------------
# Loss + Train Step
# ----------------------------

def reconstruction_loss(x_hat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    # MSE is common for normalized images; BCE if using sigmoid + Bernoulli assumption.
    return F.mse_loss(x_hat, x)

def vqvae_loss(x_hat: torch.Tensor, x: torch.Tensor, qinfo: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    rec = reconstruction_loss(x_hat, x)
    vq = qinfo["vq_loss"]
    total = rec + vq
    return {"total": total, "recon": rec, "vq": vq}


@torch.no_grad()
def sanity_check_shapes(model: nn.Module, cfg: VQVAEConfig):
    x = torch.randn(4, cfg.in_channels, 32, 32, device=cfg.device)
    x_hat, info = model(x)
    assert x_hat.shape == x.shape, f"x_hat {x_hat.shape} != x {x.shape}"
    print("OK shapes:", x.shape, "->", info["z_e"].shape, "->", x_hat.shape)


def train_one_epoch(
    model: VQVAE,
    loader: DataLoader,
    optim: torch.optim.Optimizer,
    cfg: VQVAEConfig,
) -> Dict[str, float]:
    model.train()
    totals = {"total": 0.0, "recon": 0.0, "vq": 0.0, "perplexity": 0.0}
    n = 0

    for x, *_ in loader:  # dataset may return (x,y); ignore y
        x = x.to(cfg.device)

        x_hat, info = model(x)
        losses = vqvae_loss(x_hat, x, info)

        optim.zero_grad(set_to_none=True)
        losses["total"].backward()

        # Interview prompt: ask about gradient flow; which params get grads?
        # Candidate can inspect:
        #   model.quantizer.embedding.weight.grad is not None
        optim.step()

        bs = x.size(0)
        n += bs
        totals["total"] += losses["total"].item() * bs
        totals["recon"] += losses["recon"].item() * bs
        totals["vq"] += losses["vq"].item() * bs
        totals["perplexity"] += info["perplexity"].item() * bs

    return {k: v / max(n, 1) for k, v in totals.items()}


In [11]:
# ----------------------------
# Minimal main (candidate can fill dataset)
# ----------------------------

def make_dummy_loader(cfg: VQVAEConfig, steps: int = 200) -> DataLoader:
    """
    Interview-friendly: no torchvision dependency. Produces random images in [0,1].
    Replace with CIFAR/MNIST loader in real usage.
    """
    class Dummy(torch.utils.data.Dataset):
        def __len__(self): return steps * cfg.batch_size
        def __getitem__(self, idx):
            x = torch.rand(cfg.in_channels, 32, 32)  # [0,1]
            return x, 0

    return DataLoader(
        Dummy(),
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

def main():
    cfg = VQVAEConfig()
    model = VQVAE(cfg).to(cfg.device)
    sanity_check_shapes(model, cfg)

    loader = make_dummy_loader(cfg)
    optim = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    for epoch in range(3):
        metrics = train_one_epoch(model, loader, optim, cfg)
        print(f"epoch {epoch}: " + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))

In [12]:
import math
import torch
import torch.nn.functional as F

# Import your classes here
# from vqvae import VectorQuantizer, VQVAEConfig, VQVAE

def _set_codebook(vq, vectors):
    """
    vectors: (K, D) tensor
    """
    with torch.no_grad():
        vq.embedding.weight.copy_(vectors)

def test_shapes_and_dtypes():
    # from vqvae import VectorQuantizer
    vq = VectorQuantizer(num_embeddings=8, embedding_dim=4, commitment_cost=0.25)

    z_e = torch.randn(2, 4, 3, 5)  # (B,D,H,W)
    z_q_st, info = vq(z_e)

    assert z_q_st.shape == z_e.shape
    assert info["indices"].shape == (2, 3, 5)
    assert info["vq_loss"].ndim == 0
    assert info["perplexity"].ndim == 0

def test_nearest_neighbor_indices_simple():
    """
    Make codebook = axis-aligned basis; pick latents that are clearly closest.
    """
    # from vqvae import VectorQuantizer
    D = 2
    K = 4
    vq = VectorQuantizer(num_embeddings=K, embedding_dim=D, commitment_cost=0.25)

    # embeddings: e0=(0,0), e1=(1,0), e2=(0,1), e3=(1,1)
    codebook = torch.tensor([[0.,0.],[1.,0.],[0.,1.],[1.,1.]])
    _set_codebook(vq, codebook)

    # z_e at 4 spatial positions (B=1,H=2,W=2)
    # positions:
    # (0,0): near (0,0) -> 0
    # (0,1): near (1,0) -> 1
    # (1,0): near (0,1) -> 2
    # (1,1): near (1,1) -> 3
    z = torch.tensor([
        [[[0.1, 0.9],
          [0.1, 0.9]],
         [[0.1, 0.1],
          [0.9, 0.9]]]
    ])  # shape (1,D=2,H=2,W=2)

    z_q_st, info = vq(z)
    idx = info["indices"]
    expected = torch.tensor([[[0, 1],
                              [2, 3]]])
    assert torch.equal(idx.cpu(), expected), (idx, expected)

def test_quantized_values_match_codebook():
    # from vqvae import VectorQuantizer
    vq = VectorQuantizer(num_embeddings=2, embedding_dim=3, commitment_cost=0.25)
    codebook = torch.tensor([[1.,2.,3.], [-1.,-2.,-3.]])
    _set_codebook(vq, codebook)

    # Make z_e exactly equal to embedding 0 everywhere
    z_e = codebook[0].view(1, 3, 1, 1).repeat(2, 1, 4, 5)  # (B=2,D=3,H=4,W=5)

    z_q_st, info = vq(z_e)
    # z_q_st should equal z_e numerically (since nearest is exact)
    assert torch.allclose(z_q_st, z_e, atol=0, rtol=0)

def test_losses_match_definition():
    """
    Check codebook_loss = mse(z_q, stopgrad(z_e))
          commit_loss  = beta * mse(z_e, stopgrad(z_q))
    """
    # from vqvae import VectorQuantizer
    beta = 0.5
    vq = VectorQuantizer(num_embeddings=2, embedding_dim=2, commitment_cost=beta)
    codebook = torch.tensor([[0.,0.], [2.,0.]])
    _set_codebook(vq, codebook)

    # z_e values:
    # one near (0,0), one near (2,0)
    z_e = torch.tensor([[[[0.2, 1.8]],
                         [[0.0, 0.0]]]])  # (B=1,D=2,H=1,W=2)

    z_q_st, info = vq(z_e)

    # Reconstruct z_q (not ST) from indices for exact comparison
    idx = info["indices"].view(-1)
    z_q_flat = vq.embedding(idx)  # (N,D)
    z_q = z_q_flat.view(1, 1, 2, 2).permute(0, 3, 1, 2).contiguous()  # (B,D,H,W)

    codebook_loss = F.mse_loss(z_q, z_e.detach())
    commit_loss = beta * F.mse_loss(z_e, z_q.detach())
    expected_vq = codebook_loss + commit_loss

    assert torch.allclose(info["codebook_loss"], codebook_loss)
    assert torch.allclose(info["commitment_loss"], commit_loss)
    assert torch.allclose(info["vq_loss"], expected_vq)

def test_straight_through_gradient_behavior():
    """
    Key property:
      z_q_st = z_e + (z_q - z_e).detach()
    => d(z_q_st)/d(z_e) = I (identity), so grads flow to encoder as if quantization is identity.
    """
    # from vqvae import VectorQuantizer
    vq = VectorQuantizer(num_embeddings=4, embedding_dim=3, commitment_cost=0.25)

    z_e = torch.randn(2, 3, 4, 4, requires_grad=True)
    z_q_st, info = vq(z_e)

    # Choose a simple scalar objective that depends on z_q_st
    loss = (z_q_st ** 2).mean()
    loss.backward()

    # If STE is correct, z_e.grad should not be None and should match gradient of (z_e^2).mean()
    assert z_e.grad is not None
    expected_grad = (2 * z_e.detach()) / z_e.numel()
    assert torch.allclose(z_e.grad, expected_grad, atol=1e-6, rtol=1e-5)

def test_embedding_gets_grad_from_codebook_loss():
    """
    Embedding vectors should receive gradients (from codebook loss term).
    """
    # from vqvae import VectorQuantizer
    vq = VectorQuantizer(num_embeddings=8, embedding_dim=4, commitment_cost=0.25)

    z_e = torch.randn(2, 4, 3, 3)  # encoder output (no grad needed for this test)
    z_q_st, info = vq(z_e)

    # backprop through vq_loss; embedding should get grads
    loss = info["vq_loss"]
    vq.embedding.weight.grad = None
    loss.backward()

    assert vq.embedding.weight.grad is not None
    assert torch.isfinite(vq.embedding.weight.grad).all()

def test_perplexity_extremes():
    """
    If all latents map to one code -> perplexity ~ 1
    If latents are evenly split among K codes -> perplexity ~ K
    """
    # from vqvae import VectorQuantizer
    K, D = 4, 2
    vq = VectorQuantizer(num_embeddings=K, embedding_dim=D, commitment_cost=0.25)

    # Make a simple codebook
    codebook = torch.tensor([[0.,0.],[1.,0.],[0.,1.],[1.,1.]])
    _set_codebook(vq, codebook)

    # Case 1: all near code 0
    z1 = torch.zeros(1, D, 4, 4) + 0.01
    _, info1 = vq(z1)
    assert abs(info1["perplexity"].item() - 1.0) < 1e-3

    # Case 2: evenly use all codes (16 positions -> 4 each)
    z2 = torch.zeros(1, D, 4, 4)
    targets = [codebook[0], codebook[1], codebook[2], codebook[3]] * 4  # 16 vectors
    targets = torch.stack(targets, dim=0)  # (16,D)
    z2_flat = targets.view(1, 4, 4, D).permute(0, 3, 1, 2).contiguous()
    _, info2 = vq(z2_flat)
    assert abs(info2["perplexity"].item() - K) < 1e-2

if __name__ == "__main__":
    # Run without pytest
    for fn in [
        test_shapes_and_dtypes,
        test_nearest_neighbor_indices_simple,
        test_quantized_values_match_codebook,
        test_losses_match_definition,
        test_straight_through_gradient_behavior,
        test_embedding_gets_grad_from_codebook_loss,
        test_perplexity_extremes,
    ]:
        fn()
    print("All tests passed.")

AssertionError: 

## Solutions

```python
# ----------------------------
# Vector Quantizer (candidate implements)
# ----------------------------

class VectorQuantizer(nn.Module):
    """
    Standard VQ-VAE codebook quantizer.

    Input z_e: (B, D, H, W)
    Output:
      z_q_st: quantized with straight-through gradient, same shape as z_e
      info dict: indices, losses, perplexity, etc.

    Interview TODOs:
      - compute distances efficiently
      - pick nearest embedding for each latent position
      - compute VQ loss + commitment loss
      - implement straight-through estimator: z_q_st = z_e + (z_q - z_e).detach()
    """
    def __init__(self, num_embeddings: int, embedding_dim: int, commitment_cost: float):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        # Initialize embeddings (common choices: uniform or normal)
        nn.init.uniform_(self.embedding.weight, -1.0 / num_embeddings, 1.0 / num_embeddings)

    def forward(self, z_e: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        z_e: (B, D, H, W)
        """
        assert z_e.dim() == 4, f"expected (B,D,H,W), got {tuple(z_e.shape)}"
        B, D, H, W = z_e.shape
        assert D == self.embedding_dim, f"D={D} must equal embedding_dim={self.embedding_dim}"

        # Flatten to (N, D) where N = B*H*W
        z_e_flat = z_e.permute(0, 2, 3, 1).contiguous().view(-1, D)  # (N, D)

        # ---- TODO(1): compute nearest embedding indices ----
        # Hint: squared L2 distance:
        #   ||x - e||^2 = ||x||^2 + ||e||^2 - 2 x·e
        # z_e_flat: (N,D), embed: (K,D)
        # distances: (N,K)
        # indices: (N,)
        #
        # distances = ...
        # indices = distances.argmin(dim=1)
        #
        # For interview: ask them about memory; K might be large.
        embed = self.embedding.weight  # (K, D)

        # Placeholder (wrong) — candidate replaces:
        distances = torch.cdist(z_e_flat, embed) ** 2  # (N, K) (slow but easy)
        indices = distances.argmin(dim=1)              # (N,)

        # ---- TODO(2): lookup quantized vectors ----
        z_q_flat = self.embedding(indices)             # (N, D)
        z_q = z_q_flat.view(B, H, W, D).permute(0, 3, 1, 2).contiguous()  # (B,D,H,W)

        # ---- TODO(3): losses ----
        # VQ-VAE loss terms typically:
        #   codebook_loss = ||sg[z_e] - z_q||^2
        #   commitment_loss = beta * ||z_e - sg[z_q]||^2
        # where sg[] is stop-gradient (detach).
        codebook_loss = F.mse_loss(z_q, z_e.detach())
        commitment_loss = self.commitment_cost * F.mse_loss(z_e, z_q.detach())
        vq_loss = codebook_loss + commitment_loss

        # ---- TODO(4): straight-through estimator ----
        z_q_st = z_e + (z_q - z_e).detach()

        # Optional metrics: perplexity (how many codes are used)
        one_hot = F.one_hot(indices, num_classes=self.num_embeddings).float()  # (N, K)
        avg_probs = one_hot.mean(dim=0)                                        # (K,)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        info = {
            "indices": indices.view(B, H, W),  # nicer shape
            "vq_loss": vq_loss,
            "codebook_loss": codebook_loss,
            "commitment_loss": commitment_loss,
            "perplexity": perplexity,
        }
        return z_q_st, info
```